In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import math

# 1. CONFIG (tetap minimalis)
@dataclass
class Config:
    vocab_size: int = 50257
    block_size: int = 768
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    rope_theta: float = 500000.0
    norm_eps: float = 1e-6

# 2. RMSNorm (tetap clean)
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * self.scale / (x.pow(2).mean(-1, keepdim=True) + self.eps).sqrt()

# 3. RoPE precompute (tetap optimal)
def precompute_rope_freqs(dim, max_len, theta=500000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[:dim//2].float() / dim))
    t = torch.arange(max_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)

# 4. RoPE apply (tetap robust)
def apply_rotary_emb(q, k, cos, sin):
    head_dim = q.shape[-1]
    q_real, q_imag = q[..., :head_dim//2], q[..., head_dim//2:]
    k_real, k_imag = k[..., :head_dim//2], k[..., head_dim//2:]
    
    cos = cos[:, :, :q.shape[2], :]
    sin = sin[:, :, :q.shape[2], :]
    
    q_rot = torch.cat((q_real * cos - q_imag * sin, q_real * sin + q_imag * cos), dim=-1)
    k_rot = torch.cat((k_real * cos - k_imag * sin, k_real * sin + k_imag * cos), dim=-1)
    return q_rot, k_rot

# 5. BLOCK (SwiGLU 2.7x + HAPUS explicit init — biar _init_weights handle semua)
class AmadeusZeroBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        
        # LLaMA-style SwiGLU: (8/3)*n_embd rounded to 256-multiple
        hidden_dim = int(8 * config.n_embd / 3)
        hidden_dim = ((hidden_dim + 255) // 256) * 256  # GPU-friendly alignment
        
        self.ln_1 = RMSNorm(config.n_embd, eps=config.norm_eps)
        self.ln_2 = RMSNorm(config.n_embd, eps=config.norm_eps)
        
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)  # Init di-handle di _init_weights
        
        self.mlp = nn.ModuleDict({
            'gate_proj': nn.Linear(config.n_embd, hidden_dim, bias=False),
            'up_proj': nn.Linear(config.n_embd, hidden_dim, bias=False),
            'down_proj': nn.Linear(hidden_dim, config.n_embd, bias=False),  # Init di-handle di _init_weights
        })

    def forward(self, x, cos, sin):
        x = x + self._attn_block(self.ln_1(x), cos, sin)
        x = x + self._mlp_block(self.ln_2(x))
        return x

    def _attn_block(self, x, cos, sin):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        
        q, k = apply_rotary_emb(q, k, cos, sin)
        
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

    def _mlp_block(self, x):
        gate = F.silu(self.mlp.gate_proj(x))
        up = self.mlp.up_proj(x)
        return self.mlp.down_proj(gate * up)

# 6. MAIN MODEL (FIX INIT: depth-scaled buat SEMUA output projections)
class AmadeusZeroLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict({
            'wte': nn.Embedding(config.vocab_size, config.n_embd),
            'h': nn.ModuleList([AmadeusZeroBlock(config) for _ in range(config.n_layer)]),
            'ln_f': RMSNorm(config.n_embd, eps=config.norm_eps),
        })
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.transformer.wte.weight  # Weight tying
        
        # Precompute RoPE
        dim = config.n_embd // config.n_head
        max_len = config.block_size * 2
        cos, sin = precompute_rope_freqs(dim, max_len, theta=config.rope_theta)
        self.register_buffer("cos", cos.unsqueeze(0).unsqueeze(0))
        self.register_buffer("sin", sin.unsqueeze(0).unsqueeze(0))
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        """Depth-scaled init untuk output projections (stabilin residual branches)"""
        std = 0.02
        if isinstance(module, nn.Linear):
            # Output projections (c_proj & down_proj) punya out_features = n_embd
            # Scale dengan 1/sqrt(2 * depth) sesuai LLaMA best practice
            if module.weight.size(0) == self.config.n_embd:  # out_features == n_embd
                std = 0.02 / math.sqrt(2 * self.config.n_layer)
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        _, t = idx.size()
        assert t <= self.config.block_size, f"Sequence length {t} exceeds block_size {self.config.block_size}"
        
        tok_emb = self.transformer.wte(idx)
        cos = self.cos[:, :, :t, :]
        sin = self.sin[:, :, :t, :]
        
        x = tok_emb
        for block in self.transformer.h:
            x = block(x, cos, sin)
        
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [2]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

print(f"Vocabulary size: {tokenizer.n_vocab:,}")
print(f"Special tokens: {tokenizer.special_tokens_set}")

Vocabulary size: 50,257
Special tokens: {'<|endoftext|>'}


In [4]:
conf = Config()
model = AmadeusZeroLM(conf)
# PENTING: Wrap DataParallel DULU baru pindahin ke .to(device)
if torch.cuda.device_count() > 1:
    print(f"Aktifkan {torch.cuda.device_count()} GPU T4!")
    model = nn.DataParallel(model)

device = torch.device("cuda")
model.to(device)

Aktifkan 2 GPU T4!


DataParallel(
  (module): AmadeusZeroLM(
    (transformer): ModuleDict(
      (wte): Embedding(50257, 768)
      (h): ModuleList(
        (0-11): 12 x AmadeusZeroBlock(
          (ln_1): RMSNorm()
          (ln_2): RMSNorm()
          (c_attn): Linear(in_features=768, out_features=2304, bias=False)
          (c_proj): Linear(in_features=768, out_features=768, bias=False)
          (mlp): ModuleDict(
            (gate_proj): Linear(in_features=768, out_features=2048, bias=False)
            (up_proj): Linear(in_features=768, out_features=2048, bias=False)
            (down_proj): Linear(in_features=2048, out_features=768, bias=False)
          )
        )
      )
      (ln_f): RMSNorm()
    )
    (lm_head): Linear(in_features=768, out_features=50257, bias=False)
  )
)

In [5]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name:20} | Shape: {str(param.shape):20} | Params: {param.numel():,}")

module.transformer.wte.weight | Shape: torch.Size([50257, 768]) | Params: 38,597,376
module.transformer.h.0.ln_1.scale | Shape: torch.Size([768])    | Params: 768
module.transformer.h.0.ln_2.scale | Shape: torch.Size([768])    | Params: 768
module.transformer.h.0.c_attn.weight | Shape: torch.Size([2304, 768]) | Params: 1,769,472
module.transformer.h.0.c_proj.weight | Shape: torch.Size([768, 768]) | Params: 589,824
module.transformer.h.0.mlp.gate_proj.weight | Shape: torch.Size([2048, 768]) | Params: 1,572,864
module.transformer.h.0.mlp.up_proj.weight | Shape: torch.Size([2048, 768]) | Params: 1,572,864
module.transformer.h.0.mlp.down_proj.weight | Shape: torch.Size([768, 2048]) | Params: 1,572,864
module.transformer.h.1.ln_1.scale | Shape: torch.Size([768])    | Params: 768
module.transformer.h.1.ln_2.scale | Shape: torch.Size([768])    | Params: 768
module.transformer.h.1.c_attn.weight | Shape: torch.Size([2304, 768]) | Params: 1,769,472
module.transformer.h.1.c_proj.weight | Shape: t

In [6]:
def count_parameters(model):
    # Sum semua elemen di tiap parameter yang butuh gradien
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'OtterLM Parameter: {count_parameters(model):,}')

OtterLM Parameter: 123,551,232


In [7]:
conf = Config()
assert isinstance(conf.vocab_size, int), "CONFIG Broken!"
print("Config OK:", conf.vocab_size)  

Config OK: 50257


In [8]:
from huggingface_hub import hf_hub_download
import pandas as pd
# 2. Download the correct parquet file from the repository
repo_id = "karpathy/tinystories-gpt4-clean"
filename = "tinystories_gpt4_clean.parquet"

print(f"Downloading {filename} from {repo_id}...")
downloaded_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    repo_type="dataset"
)

# 3. Process the Parquet file using pandas
# Reading the binary format correctly into a structured DataFrame
df = pd.read_parquet(downloaded_path)

# Extracting the text column (assuming 'text' or 'story' is the column name)
# If the column name is different, we grab the first column automatically
text_column = df.columns[0]
stories = df[text_column].tolist()

print("First story (300 chars):\n")
story = stories[0]
print(story.strip()[:300], "...")

print(f"\nTotal number of stories: {len(stories):,}")

tinystories_gpt4_clean.parquet:   0%|          | 0.00/673M [00:00<?, ?B/s]

First story (300 chars):

Once there was a little boy named Jack. He was only three years old and had lots of things he wanted to do. One day he saw something very special in the park - a big wheel! It was big and bright and looked very inviting.
Jack wanted to get on the wheel, so he ran to it. He put his hands on it and ge ...

Total number of stories: 2,732,634
